In [42]:
import numpy as np 
import pandas as pd
import seaborn as sns
import sys
import os
import importlib  # <-- 1. Added missing importlib

# Go up one level ('..') from 'notebooks/' to the root directory, then into 'backend'
backend_path = os.path.abspath(os.path.join('..', 'backend'))
if backend_path not in sys.path:
    sys.path.append(backend_path)

import src.utils
import src.clean

# Reload to clear cache and pull latest edits from disk
importlib.reload(src.utils)
importlib.reload(src.clean)

# Import functions fresh (using correct names from utils.py)
from src.utils import clean_crime_type, clean_property_loss_usd_val,clean_incident_datetime
from src.clean import load_and_clean_data

# Pandas display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', None)

print("Setup complete and modules loaded successfully!")

Setup complete and modules loaded successfully!


In [77]:
df = pd.read_csv("../data/raw/Chicago_Crimes_2001_to_2004.csv", on_bad_lines='skip')

C:\Users\dsita\AppData\Local\Temp\ipykernel_16208\2933981705.py:1: DtypeWarning: Columns (0: Y Coordinate, 1: Latitude) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/raw/Chicago_Crimes_2001_to_2004.csv", on_bad_lines='skip')


In [75]:
# import pandas as pd
# import glob

# # Find all CSV files in your raw data directory
# file_paths = glob.glob("../data/raw/Chicago_Crimes_*.csv")

# df_list = []
# for file in file_paths:
#     print(f"Loading {file}...")
#     temp_df = pd.read_csv(file, on_bad_lines='skip', low_memory=False)
#     df_list.append(temp_df)

# # Concatenate everything into one massive dataframe
# df = pd.concat(df_list, ignore_index=True)
# print(f"Combined dataset shape: {df.shape}")

Loading ../data/raw\Chicago_Crimes_2001_to_2004.csv...
Loading ../data/raw\Chicago_Crimes_2005_to_2007.csv...
Loading ../data/raw\Chicago_Crimes_2008_to_2011.csv...
Loading ../data/raw\Chicago_Crimes_2012_to_2017.csv...
Combined dataset shape: (7941282, 23)


In [80]:
df.info(show_counts=True)

<class 'pandas.DataFrame'>
RangeIndex: 1923515 entries, 0 to 1923514
Data columns (total 23 columns):
 #   Column                Non-Null Count    Dtype  
---  ------                --------------    -----  
 0   Unnamed: 0            1923515 non-null  int64  
 1   ID                    1923515 non-null  int64  
 2   Case Number           1923515 non-null  str    
 3   Date                  1923515 non-null  str    
 4   Block                 1923515 non-null  str    
 5   IUCR                  1923515 non-null  str    
 6   Primary Type          1923515 non-null  str    
 7   Description           1923515 non-null  str    
 8   Location Description  1923499 non-null  str    
 9   Arrest                1923515 non-null  bool   
 10  Domestic              1923515 non-null  bool   
 11  Beat                  1923515 non-null  int64  
 12  District              1923513 non-null  float64
 13  Ward                  1223383 non-null  float64
 14  Community Area        1223268 non-null  float

In [76]:
df.head()

,Unnamed: 0,ID,Case Number,Date,Block,IUCR,Primary Type,Description,Location Description,Arrest,Domestic,Beat,District,Ward,Community Area,FBI Code,X Coordinate,Y Coordinate,Year,Updated On,Latitude,Longitude,Location
0,879,4786321,HM399414,01/01/2004 12:01:00 AM,082XX S COLES AVE,0840,THEFT,FINANCIAL ID THEFT: OVER $300,RESIDENCE,False,False,424,4.0,7.0,46.0,06,NaN,NaN,2004.0,08/17/2015 03:03:40 PM,NaN,NaN,NaN
1,2544,4676906,HM278933,03/01/2003 12:00:00 AM,004XX W 42ND PL,2825,OTHER OFFENSE,HARASSMENT BY TELEPHONE,RESIDENCE,False,True,935,9.0,11.0,61.0,26,1173974.0,1876757.0,2003.0,04/15/2016 08:55:02 AM,41.817229156,-87.637328,"(41.817229156, -87.637328162)"
2,2919,4789749,HM402220,06/20/2004 11:00:00 AM,025XX N KIMBALL AVE,1752,OFFENSE INVOLVING CHILDREN,AGG CRIM SEX ABUSE FAM MEMBER,RESIDENCE,False,False,1413,14.0,35.0,22.0,20,NaN,NaN,2004.0,08/17/2015 03:03:40 PM,NaN,NaN,NaN
3,2927,4789765,HM402058,12/30/2004 08:00:00 PM,045XX W MONTANA ST,0840,THEFT,FINANCIAL ID THEFT: OVER $300,OTHER,False,False,2521,25.0,31.0,20.0,06,NaN,NaN,2004.0,08/17/2015 03:03:40 PM,NaN,NaN,NaN
4,3302,4677901,HM275615,05/01/2003 01:00:00 AM,111XX S NORMAL AVE,0841,THEFT,FINANCIAL ID THEFT:$300 &UNDER,RESIDENCE,False,False,2233,22.0,34.0,49.0,06,1174948.0,1831051.0,2003.0,04/15/2016 08:55:02 AM,41.691784636,-87.635116,"(41.691784636, -87.635115968)"


In [98]:
df.Date.isna().sum()

np.int64(0)

In [81]:
df.Arrest.value_counts()

Arrest
False    1354640
True      568875
Name: count, dtype: int64

In [82]:
df.Arrest.isna().sum()

np.int64(0)

In [83]:
df.Domestic.value_counts()

Domestic
False    1693509
True      230006
Name: count, dtype: int64

In [84]:
df.Domestic.isna().sum()

np.int64(0)

In [90]:
print(df["Location Description"].value_counts().head(20))

Location Description
STREET                            584118
RESIDENCE                         349772
APARTMENT                         144214
SIDEWALK                          140802
OTHER                              78132
PARKING LOT/GARAGE(NON.RESID.)     59123
SCHOOL, PUBLIC, BUILDING           44898
ALLEY                              43061
RESIDENCE-GARAGE                   38931
RESIDENCE PORCH/HALLWAY            34843
GROCERY FOOD STORE                 31091
SMALL RETAIL STORE                 30541
CHA PARKING LOT/GROUNDS            30117
RESTAURANT                         27141
DEPARTMENT STORE                   22438
VEHICLE NON-COMMERCIAL             20757
CHA APARTMENT                      19283
GAS STATION                        19102
COMMERCIAL / BUSINESS OFFICE       18944
PARK PROPERTY                      14121
Name: count, dtype: int64


In [91]:
value_counts = df["Location Description"].value_counts()
rare_locations = value_counts[value_counts < 10]
print(f"Number of rare location categories with < 10 occurrences: {len(rare_locations)}")

Number of rare location categories with < 10 occurrences: 42


In [92]:
# Define a frequency threshold
threshold = 100
counts = df["Location Description"].value_counts()

# Create a mask for rare categories
to_collapse = counts[counts < threshold].index

# Replace rare categories with 'OTHER'
df["Location Description Cleaned"] = df["Location Description"].replace(to_collapse, "OTHER")

print(f"Reduced unique locations from {df['Location Description'].nunique()} to {df['Location Description Cleaned'].nunique()}")

Reduced unique locations from 144 to 79


In [93]:
import pandas as pd

def map_location_category(location):
    if pd.isna(location):
        return "UNKNOWN"
    
    loc = str(location).upper().strip()
    
    # 1. Residential Spaces
    if any(k in loc for k in ["RESIDENCE", "APARTMENT", "CHA APARTMENT", "HOME"]):
        return "RESIDENTIAL"
    
    # 2. Public Ways & Open Spaces
    elif any(k in loc for k in ["STREET", "SIDEWALK", "ALLEY", "PARK", "DRIVEWAY", "PATH", "BRIDGE"]):
        return "PUBLIC_WAY"
    
    # 3. Commercial & Retail Businesses
    elif any(k in loc for k in ["STORE", "RESTAURANT", "BAR", "GAS STATION", "BANK", "HOTEL", 
                                "TAXI", "COMMERCIAL", "TAVERN", "LIDO", "MARKET", "SHOP"]):
        return "COMMERCIAL_RETAIL"
    
    # 4. Vehicles & Parking
    elif any(k in loc for k in ["AUTO", "VEHICLE", "PARKING", "BUS", "TRAIN", "CAB", "AIRCRAFT"]):
        return "VEHICLE_PARKING"
    
    # 5. Schools, Government, & Institutional
    elif any(k in loc for k in ["SCHOOL", "UNIVERSITY", "COLLEGE", "POLICE", "GOVERNMENT", 
                                "HOSPITAL", "LIBRARY", "CHURCH", "JAIL"]):
        return "INSTITUTIONAL"
        
    # 6. Everything else falls into a consolidated bucket
    else:
        return "OTHER"

# Apply the mapping to create a clean, low-cardinality feature
df['Location_Macro'] = df['Location Description'].apply(map_location_category)

# Check the distribution of your new macro-categories
print(df['Location_Macro'].value_counts())

Location_Macro
PUBLIC_WAY           880908
RESIDENTIAL          591224
COMMERCIAL_RETAIL    222861
OTHER                135662
INSTITUTIONAL         74409
VEHICLE_PARKING       18435
UNKNOWN                  16
Name: count, dtype: int64


In [94]:
df['Location_Macro'].isna().sum()

np.int64(0)

In [87]:
df["Primary Type"].value_counts().sort_index()

Primary Type
ARSON                                 3828
ASSAULT                             121417
BATTERY                             363538
BURGLARY                            101902
CRIM SEXUAL ASSAULT                   7079
CRIMINAL DAMAGE                     220298
CRIMINAL TRESPASS                    57770
DECEPTIVE PRACTICE                   55751
DOMESTIC VIOLENCE                        2
GAMBLING                              4051
HOMICIDE                              3172
INTERFERENCE WITH PUBLIC OFFICER      1701
INTIMIDATION                          1312
KIDNAPPING                            3022
LIQUOR LAW VIOLATION                  5384
MOTOR VEHICLE THEFT                  98924
NARCOTICS                           210600
NON-CRIMINAL                             1
OBSCENITY                               74
OFFENSE INVOLVING CHILDREN           11181
OTHER NARCOTIC VIOLATION                22
OTHER OFFENSE                       123753
PROSTITUTION                         2566

AttributeError: 'DataFrame' object has no attribute 'date'